In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, HBox, HTML, Layout
from IPython.display import display

# ============================================================
# BILINEAR TRANSFORMATION MAPPING
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.bl-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.bl-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:11px 15px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.bl-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:11px 14px;
    border-radius:0 0 8px 8px;
    font-size:15px;
    line-height:1.55;
    margin-bottom:9px;
}

.bl-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:10px 12px;
    margin-bottom:8px;
    font-size:14.5px;
    line-height:1.50;
}

.bl-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:15.5px;
    margin-bottom:6px;
}

.bl-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.bl-col{
    flex:1;
    min-width:0;
}

.widget-label{
    font-size:14px !important;
}

.jupyter-widgets input{
    font-size:13.5px !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="bl-root">

<div class="bl-header">
Bilinear Transformation: Mapping from the s-Plane to the z-Plane
</div>

<div class="bl-doc">

The bilinear transformation maps the continuous-time complex variable
<b>s = σ+jΩ</b> to the discrete-time complex variable z through

<div style="text-align:center;font-size:16px;margin:9px 0;">
<b>
z =
(1+sT<sub>s</sub>/2) /
(1-sT<sub>s</sub>/2).
</b>
</div>

Equivalently,

<div style="text-align:center;font-size:16px;margin:9px 0;">
<b>
s =
(2/T<sub>s</sub>)
(1-z<sup>-1</sup>) /
(1+z<sup>-1</sup>).
</b>
</div>

The transformation maps the complete left half-plane of the s-plane into the
interior of the unit circle and therefore preserves stability.

For points on the imaginary axis, <b>s=jΩ</b>, the image lies on the unit circle and

<div style="text-align:center;font-size:16px;margin:9px 0;">
<b>
ω =
2 tan<sup>-1</sup>(ΩT<sub>s</sub>/2).
</b>
</div>

Thus the complete analog frequency axis
<b>-∞ &lt; Ω &lt; ∞</b>
is mapped one-to-one onto
<b>-π &lt; ω &lt; π</b>.

Unlike impulse invariance, no multiple analog-frequency intervals are mapped onto
the same digital-frequency interval, and therefore the bilinear transformation does
not produce aliasing.

The price for this one-to-one mapping is <b>frequency warping</b>: the relationship
between Ω and ω is nonlinear.

</div>

</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

Ts_slider = FloatSlider(value=0.50,min=0.10,max=1.00,step=0.01,description='Ts:',continuous_update=True,readout_format='.2f',style={'description_width':'30px'},layout=Layout(width='205px'))

sigma_slider = FloatSlider(value=-1.00,min=-3.00,max=1.00,step=0.05,description='σ:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='245px'))

Omega_slider = FloatSlider(value=2.00,min=-10.00,max=10.00,step=0.10,description='Ω:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='260px'))

control_title = HTML('<div class="bl-title" style="margin:0;">Transformation parameters</div>',layout=Layout(width='170px'))

controls = HBox([
    control_title,
    Ts_slider,
    sigma_slider,
    Omega_slider
],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='9px 12px',margin='0 0 8px 0',align_items='center'))

info = HTML(layout=Layout(width=CONTENT_WIDTH,margin='0 0 8px 0'))

# ============================================================
# FIGURE — CREATED ONCE
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.6))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. s-PLANE
# ============================================================

ax1.axhline(0,color='black',linewidth=0.8)
ax1.axvline(0,color='black',linewidth=0.8)
ax1.axvspan(-3.5,0,alpha=0.05,label='Stable half-plane')

selected_s, = ax1.plot([],[],'ro',markersize=7,label='Selected point')

ax1.set_xlim(-3.5,1.5)
ax1.set_ylim(-11,11)

ax1.set_title('Selected Point in the s-Plane')
ax1.set_xlabel(r'$\Re\{s\}$')
ax1.set_ylabel(r'$\Im\{s\}$')

ax1.grid(True,linestyle=':',alpha=0.25)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 2. z-PLANE
# ============================================================

theta = np.linspace(0,2*np.pi,1000)

ax2.axhline(0,color='black',linewidth=0.8)
ax2.axvline(0,color='black',linewidth=0.8)

ax2.plot(np.cos(theta),np.sin(theta),'--',linewidth=1.1,label='Unit circle')

mapped_z, = ax2.plot([],[],'ro',markersize=7,label='Mapped point')

# Fixed limits chosen from the complete slider ranges.
# For sigma = 1, Omega = 0 and Ts = 1, the maximum |z| is 3.

ax2.set_xlim(-3.2,3.2)
ax2.set_ylim(-3.2,3.2)
ax2.set_aspect('equal',adjustable='box')

ax2.set_title('Mapped Point in the z-Plane')
ax2.set_xlabel(r'$\Re\{z\}$')
ax2.set_ylabel(r'$\Im\{z\}$')

ax2.grid(True,linestyle=':',alpha=0.25)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 3. NONLINEAR FREQUENCY MAPPING
# ============================================================

Omega_curve = np.linspace(-20,20,4000)

warping_line, = ax3.plot([],[],color='red',linewidth=1.4,label='Bilinear transformation')

ax3.axhline(np.pi,linestyle='--',linewidth=1.0,label=r'$\omega=\pi$')
ax3.axhline(-np.pi,linestyle='--',linewidth=1.0,label=r'$\omega=-\pi$')

selected_frequency, = ax3.plot([],[],'ro',markersize=5,label='Selected frequency')

ax3.set_xlim(-20,20)
ax3.set_ylim(-3.5,3.5)

ax3.set_title('Nonlinear Frequency Mapping')
ax3.set_xlabel(r'Analog frequency $\Omega$')
ax3.set_ylabel(r'Digital frequency $\omega$')

ax3.grid(True,linestyle=':',alpha=0.25)
ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 4. LINEAR VS BILINEAR FREQUENCY MAPPING
# ============================================================

comparison_bilinear, = ax4.plot([],[],color='red',linewidth=1.4,label='Bilinear transformation')
comparison_linear, = ax4.plot([],[],'--',linewidth=1.2,label=r'Linear mapping $\omega=\Omega T_s$')

ax4.axhline(np.pi,linestyle=':',linewidth=1.0)
ax4.axhline(-np.pi,linestyle=':',linewidth=1.0)

ax4.set_xlim(-10,10)
ax4.set_ylim(-3.5,3.5)

ax4.set_title('Frequency Warping')
ax4.set_xlabel(r'Analog frequency $\Omega$')
ax4.set_ylabel(r'Digital frequency $\omega$')

ax4.grid(True,linestyle=':',alpha=0.25)
ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.30,hspace=0.62)

# ============================================================
# UPDATE
# ============================================================

def update_mapping(change=None):

    Ts = Ts_slider.value
    sigma = sigma_slider.value
    Omega = Omega_slider.value

    # --------------------------------------------------------
    # Selected point in the s-plane
    # --------------------------------------------------------

    s = sigma+1j*Omega

    # --------------------------------------------------------
    # Bilinear transformation
    # --------------------------------------------------------

    z = (1+s*Ts/2)/(1-s*Ts/2)

    radius = np.abs(z)

    # --------------------------------------------------------
    # Frequency mapping for the imaginary axis
    # --------------------------------------------------------

    omega_selected = 2*np.arctan(Omega*Ts/2)

    omega_curve = 2*np.arctan(Omega_curve*Ts/2)

    # --------------------------------------------------------
    # Linear mapping for comparison
    # --------------------------------------------------------

    Omega_compare = np.linspace(-10,10,3000)

    omega_bilinear_compare = 2*np.arctan(Omega_compare*Ts/2)

    omega_linear_compare = Omega_compare*Ts

    # --------------------------------------------------------
    # Update plots
    # --------------------------------------------------------

    selected_s.set_data([sigma],[Omega])

    mapped_z.set_data([np.real(z)],[np.imag(z)])

    warping_line.set_data(Omega_curve,omega_curve)

    selected_frequency.set_data([Omega],[omega_selected])

    comparison_bilinear.set_data(Omega_compare,omega_bilinear_compare)

    comparison_linear.set_data(Omega_compare,omega_linear_compare)

    # --------------------------------------------------------
    # Classification
    # --------------------------------------------------------

    if sigma < 0:

        analog_region = 'LEFT HALF-PLANE'
        digital_region = 'INSIDE UNIT CIRCLE'

    elif sigma > 0:

        analog_region = 'RIGHT HALF-PLANE'
        digital_region = 'OUTSIDE UNIT CIRCLE'

    else:

        analog_region = 'IMAGINARY AXIS'
        digital_region = 'UNIT CIRCLE'

    # --------------------------------------------------------
    # Numerical information
    # --------------------------------------------------------

    info.value = f"""
    <div class="bl-root">

    <div class="bl-box">

    <div class="bl-title">Current bilinear transformation</div>

    <div class="bl-cols">

    <div class="bl-col">
    Sampling period:<br>
    <b>T<sub>s</sub> = {Ts:.2f}</b>
    <br><br>
    Analog point:<br>
    <b>s = {sigma:.2f} {Omega:+.2f}j</b>
    </div>

    <div class="bl-col">
    Mapped point:<br>
    <b>z = {np.real(z):.6f} {np.imag(z):+.6f}j</b>
    <br><br>
    Magnitude:<br>
    <b>|z| = {radius:.6f}</b>
    </div>

    <div class="bl-col">
    Analog frequency:<br>
    <b>Ω = {Omega:.4f}</b>
    <br><br>
    Mapped digital frequency:<br>
    <b>ω = {omega_selected:.6f}</b>
    </div>

    <div class="bl-col">
    s-plane region:<br>
    <b>{analog_region}</b>
    <br><br>
    z-plane region:<br>
    <b>{digital_region}</b>
    </div>

    </div>

    </div>

    <div class="bl-box">

    <div class="bl-title">One-to-one frequency mapping</div>

    The complete analog-frequency axis

    <div style="text-align:center;font-size:15.5px;margin:7px 0;">
    <b>-∞ &lt; Ω &lt; ∞</b>
    </div>

    is mapped exactly once onto

    <div style="text-align:center;font-size:15.5px;margin:7px 0;">
    <b>-π &lt; ω &lt; π.</b>
    </div>

    Consequently, unlike impulse invariance, the bilinear transformation does
    <b>not</b> map several different analog-frequency bands onto the same
    digital-frequency interval and therefore does not generate aliasing.

    </div>

    </div>
    """

    fig.canvas.draw_idle()

# ============================================================
# EVENTS
# ============================================================

Ts_slider.observe(update_mapping,names='value')
sigma_slider.observe(update_mapping,names='value')
Omega_slider.observe(update_mapping,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(info)
display(controls)
display(fig.canvas)

# ============================================================
# INITIAL UPDATE
# ============================================================

update_mapping()